# TP – CAH et DBSCAN sur atelier_connecte
Note : la colonne `regime_reference` est absente du fichier fourni, l'analyse des régimes n'est donc pas faite.

In [ ]:
import numpy as np, pandas as pd, matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from sklearn.cluster import AgglomerativeClustering, DBSCAN, KMeans
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.neighbors import NearestNeighbors
from sklearn.datasets import make_blobs, make_moons, make_circles
pd.set_option("display.width",200)
SEED=42
df=pd.read_csv("tp/data/atelier_connecte.csv")
feats=["temperature","vibration","courant","cadence"]
Xs=StandardScaler().fit_transform(SimpleImputer(strategy="median").fit_transform(df[feats]))
print("taux defaut",df.defaut.mean().round(3), df.defaut.sum())
print(df.groupby("machine_id").defaut.agg(["mean","sum"]).round(3))

## 1. Familles de machines

In [ ]:
prof=pd.DataFrame(Xs,columns=feats).assign(machine_id=df.machine_id.values).groupby("machine_id").mean()
print(prof.round(2))
Pm=StandardScaler().fit_transform(prof); Z=linkage(Pm,"ward")
plt.figure(figsize=(8,4)); dendrogram(Z,labels=prof.index.tolist()); plt.title("Dendrogramme des machines (Ward)"); plt.ylabel("Distance de Ward"); plt.tight_layout(); plt.savefig("dendro_machines.png",dpi=120); plt.close()
prof["famille"]=fcluster(Z,3,"maxclust"); print(prof.famille); print(prof.groupby("famille")[feats].mean().round(2))
print("Z",Z[:,2].round(2))

## 2. Comparaison des liaisons

In [ ]:
rng=np.random.default_rng(SEED); S=Xs[rng.choice(len(Xs),300,replace=False)]
labs={m:AgglomerativeClustering(n_clusters=4,linkage=m).fit_predict(S) for m in ["single","complete","average","ward"]}
print(pd.DataFrame([{"liaison":m,"eff":sorted(np.bincount(l),reverse=True),"sil":round(silhouette_score(S,l),3),"ARI":round(adjusted_rand_score(labs["ward"],l),3)} for m,l in labs.items()]))

## 3. Lecture de la coupure

In [ ]:
Zs=linkage(S,"ward"); h=Zs[:,2]; g=np.diff(h)
plt.figure(figsize=(9,4)); dendrogram(Zs,no_labels=True,truncate_mode="lastp",p=20); plt.title("Dendrogramme échantillon (Ward)"); plt.tight_layout(); plt.savefig("dendro_echantillon.png",dpi=120); plt.close()
print("hauteurs 8 dernieres",h[-8:].round(2)); print("marches",g[-8:].round(2))
i=np.argmax(g); print("max marche",g[i].round(2),"entre fusion",i,"k=",len(S)-(i+1))

## 4. Choix de eps

In [ ]:
d=np.sort(NearestNeighbors(n_neighbors=6).fit(Xs).kneighbors(Xs)[0][:,-1])  # 5e voisin hors soi
d4=np.sort(NearestNeighbors(n_neighbors=5).fit(Xs).kneighbors(Xs)[0][:,-1])
print("q 5e voisin strict",{q:round(float(np.quantile(d,q/100)),3) for q in (90,95,99)})
print("q conv sklearn (soi inclus)",{q:round(float(np.quantile(d4,q/100)),3) for q in (90,95,99)})
plt.figure(figsize=(6,4)); plt.plot(d); [plt.axhline(np.quantile(d,q/100),ls="--",c="gray") for q in (.9*100,95,99)]; plt.xlabel("points triés"); plt.ylabel("distance au 5e voisin"); plt.title("Graphe des k-distances (k=5)"); plt.tight_layout(); plt.savefig("kdist.png",dpi=120); plt.close()
for e in [.3,.4,.5,.6,.8]:
    l=DBSCAN(eps=e,min_samples=5).fit_predict(Xs); print(e,len(set(l))-(-1 in l),round((l==-1).mean(),3),(l==-1).sum())

## 5. Analyse du bruit

In [ ]:
l=DBSCAN(eps=.5,min_samples=5).fit_predict(Xs); df["bruit"]=l==-1
print(df.groupby("bruit").defaut.agg(["mean","sum","size"]).round(3))
print(pd.crosstab(df.machine_id,df.bruit))
print(pd.crosstab(df.machine_id,l))
m=l!=-1; print("sil hors bruit",silhouette_score(Xs[m],l[m]) if len(set(l[m]))>1 else None, set(l))
print(pd.Series(l).value_counts())
for a in ["alarme_temperature","alarme_vibration","alarme_courant","alarme_cadence"]: print(a,df.groupby("bruit")[a].mean().round(3).tolist())
print(df[feats].isna().any(axis=1).groupby(df.bruit).mean())

## 6. Formes synthétiques

In [ ]:
sets={"blobs":make_blobs(300,centers=3,random_state=SEED),"moons":make_moons(300,noise=.06,random_state=SEED),"circles":make_circles(300,noise=.05,factor=.5,random_state=SEED)}
ks={"blobs":3,"moons":2,"circles":2}
rows=[]
for n,(X,y) in sets.items():
    X=StandardScaler().fit_transform(X);k=ks[n]
    P={"K-means":KMeans(k,n_init=10,random_state=SEED).fit_predict(X),"CAH Ward":AgglomerativeClustering(k,linkage="ward").fit_predict(X),"CAH simple":AgglomerativeClustering(k,linkage="single").fit_predict(X)}
    for e in [.1,.15,.2,.3,.5,.8]: P[f"DBSCAN eps={e}"]=DBSCAN(eps=e,min_samples=5).fit_predict(X)
    for a,p in P.items(): rows.append({"jeu":n,"methode":a,"ARI":round(adjusted_rand_score(y,p),3)})
print(pd.DataFrame(rows).pivot(index="methode",columns="jeu",values="ARI"))